# 02 – Silver: real column names, types, cleaning
Names come from the CREATE TABLE statements in instawdb.sql (Purchasing.PurchaseOrderDetail / PurchaseOrderHeader / Vendor).

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# column order of the CSV = column order of CREATE TABLE.
# LineTotal / StockedQty / TotalDue are computed columns in SQL Server: the CSV may or may not contain them.
SPECS = {
  "stg_purchase_order_detail": dict(
    full=["PurchaseOrderID","PurchaseOrderDetailID","DueDate","OrderQty","ProductID","UnitPrice",
          "LineTotal","ReceivedQty","RejectedQty","StockedQty","ModifiedDate"],
    short=["PurchaseOrderID","PurchaseOrderDetailID","DueDate","OrderQty","ProductID","UnitPrice",
           "ReceivedQty","RejectedQty","ModifiedDate"],
    types={"PurchaseOrderID":"int","PurchaseOrderDetailID":"int","DueDate":"timestamp","OrderQty":"int",
           "ProductID":"int","UnitPrice":"decimal(19,4)","LineTotal":"decimal(19,4)",
           "ReceivedQty":"decimal(8,2)","RejectedQty":"decimal(8,2)","StockedQty":"decimal(8,2)",
           "ModifiedDate":"timestamp"}),
  "stg_purchase_order_header": dict(
    full=["PurchaseOrderID","RevisionNumber","Status","EmployeeID","VendorID","ShipMethodID","OrderDate",
          "ShipDate","SubTotal","TaxAmt","Freight","TotalDue","ModifiedDate"],
    short=["PurchaseOrderID","RevisionNumber","Status","EmployeeID","VendorID","ShipMethodID","OrderDate",
           "ShipDate","SubTotal","TaxAmt","Freight","ModifiedDate"],
    types={"PurchaseOrderID":"int","RevisionNumber":"int","Status":"int","EmployeeID":"int","VendorID":"int",
           "ShipMethodID":"int","OrderDate":"timestamp","ShipDate":"timestamp","SubTotal":"decimal(19,4)",
           "TaxAmt":"decimal(19,4)","Freight":"decimal(19,4)","TotalDue":"decimal(19,4)",
           "ModifiedDate":"timestamp"}),
  "dim_vendor": dict(
    full=["BusinessEntityID","AccountNumber","Name","CreditRating","PreferredVendorStatus","ActiveFlag",
          "PurchasingWebServiceURL","ModifiedDate"],
    short=None,
    types={"BusinessEntityID":"int","CreditRating":"int","PreferredVendorStatus":"boolean",
           "ActiveFlag":"boolean","ModifiedDate":"timestamp"}),
}

def to_silver(table):
    spec = SPECS[table]
    df = spark.table(f"{CATALOG}.bronze.{table}")
    n = len(df.columns)
    if n == len(spec["full"]):
        names = spec["full"]
    elif spec["short"] and n == len(spec["short"]):
        names = spec["short"]
    else:
        raise ValueError(f"{table}: {n} columns in bronze - compare with CREATE TABLE in instawdb.sql")
    df = df.toDF(*names)
    for c, t in spec["types"].items():
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(t))
    # computed columns missing in the CSV -> calculate them
    if table == "stg_purchase_order_detail":
        if "LineTotal" not in df.columns:
            df = df.withColumn("LineTotal", (F.col("OrderQty") * F.col("UnitPrice")).cast("decimal(19,4)"))
        if "StockedQty" not in df.columns:
            df = df.withColumn("StockedQty", (F.col("ReceivedQty") - F.col("RejectedQty")).cast("decimal(8,2)"))
    if table == "stg_purchase_order_header" and "TotalDue" not in df.columns:
        df = df.withColumn("TotalDue", (F.col("SubTotal") + F.col("TaxAmt") + F.col("Freight")).cast("decimal(19,4)"))
    return df

In [0]:
vendor = to_silver("dim_vendor")
header = to_silver("stg_purchase_order_header")
detail = to_silver("stg_purchase_order_detail")

# homework rule: on stg_purchase_order_detail drop rows with NULL ProductID or negative LineTotal
before = detail.count()
detail = detail.filter(F.col("ProductID").isNotNull() & (F.col("LineTotal") >= 0))
print("detail rows dropped:", before - detail.count())

for df, name in [(vendor, "dim_vendor"), (header, "stg_purchase_order_header"), (detail, "stg_purchase_order_detail")]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
      .saveAsTable(f"{CATALOG}.silver.{name}")
    print(name, df.count())
    display(df.limit(3))

detail rows dropped: 0
dim_vendor 104


BusinessEntityID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL,ModifiedDate
1492,AUSTRALI0001,Australia Bike Retailer,1,true,true,null,2022-12-22T00:00:00.000Z
1494,ALLENSON0001,Allenson Cycles,2,true,true,null,2022-04-24T00:00:00.000Z
1496,ADVANCED0001,Advanced Bicycles,1,true,true,null,2022-04-24T00:00:00.000Z


stg_purchase_order_header 4012


PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ShipMethodID,OrderDate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,ModifiedDate
1,5,4,258,1580,3,2022-04-15T00:00:00.000Z,2022-04-24T00:00:00.000Z,201.0400,16.0832,5.0260,222.1492,2022-04-24T00:00:00.000Z
2,5,1,254,1496,5,2022-04-15T00:00:00.000Z,2022-04-24T00:00:00.000Z,272.1015,21.7681,6.8025,300.6721,2022-04-24T00:00:00.000Z
3,5,4,257,1494,2,2022-04-15T00:00:00.000Z,2022-04-24T00:00:00.000Z,8847.3000,707.7840,221.1825,9776.2665,2022-04-24T00:00:00.000Z


stg_purchase_order_detail 8845


PurchaseOrderID,PurchaseOrderDetailID,DueDate,OrderQty,ProductID,UnitPrice,LineTotal,ReceivedQty,RejectedQty,StockedQty,ModifiedDate
1,1,2022-04-29T00:00:00.000Z,4,1,50.2600,201.0400,3.00,0.00,3.00,2022-04-22T00:00:00.000Z
2,2,2022-04-29T00:00:00.000Z,3,359,45.1200,135.3600,3.00,0.00,3.00,2022-04-22T00:00:00.000Z
2,3,2022-04-29T00:00:00.000Z,3,360,45.5805,136.7415,3.00,0.00,3.00,2022-04-22T00:00:00.000Z
